# AstroAI: Kepler Light Curve Exploration & Visualization

This notebook provides a visual exploration of the Kepler dataset downloaded in the `debo_model` workspace. It plots the raw vs. preprocessed light curves and calculates key transit parameters (depth, period, duration).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path to resolve debo_model imports
sys.path.append(os.path.dirname(os.path.abspath('')))

from debo_model.debo_preprocessing import preprocess_light_curve, align_sequence_length
from preprocessing.filters import estimate_noise

## 1. Load Dataset Index

Let's load `dataset_index.csv` to see what files are downloaded and check the class distribution.

In [ ]:
index_path = "dataset_index.csv"
if not os.path.exists(index_path):
    # Try relative path if running from root
    index_path = "debo_model/dataset_index.csv"

df = pd.read_csv(index_path)
print(f"Total downloaded curves: {len(df)}")
print("\nClass counts:")
print(df["label"].value_counts())

## 2. Visualize Raw vs. Preprocessed Light Curves

Let's select one positive example (`transit`) and one negative example (`stellar_eclipse` or `centroid_offset`), load their `.npz` files, and compare them before and after preprocessing.

In [ ]:
# Find one transit and one non-transit sample
transit_sample = df[df["label"] == "transit"].iloc[0]
non_transit_sample = df[df["label"] != "transit"].iloc[0]

def plot_comparison(sample, title):
    filepath = sample["file"]
    if not os.path.exists(filepath):
        # Try correcting relative path
        filepath = os.path.join("..", filepath)
        
    data = np.load(filepath)
    time = data["time"]
    raw_flux = data["flux"]
    
    # Run preprocessing steps
    preprocessed_flux = preprocess_light_curve(raw_flux)
    aligned_flux = align_sequence_length(preprocessed_flux, target_len=2000)
    
    # Plot side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Raw Plot
    axes[0].plot(time, raw_flux, color="crimson", alpha=0.6, label="Raw")
    axes[0].set_title(f"{title} (Raw Light Curve)", fontsize=12)
    axes[0].set_xlabel("Time (Days)", fontsize=10)
    axes[0].set_ylabel("Flux", fontsize=10)
    axes[0].grid(True, linestyle="--", alpha=0.5)
    
    # Preprocessed & Aligned Plot
    axes[1].plot(aligned_flux, color="dodgerblue", alpha=0.8, label="Cleaned & Aligned")
    axes[1].set_title(f"{title} (Cleaned & Aligned - 2000 pts)", fontsize=12)
    axes[1].set_xlabel("Index", fontsize=10)
    axes[1].set_ylabel("Normalized Relative Flux", fontsize=10)
    axes[1].grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.show()

plot_comparison(transit_sample, f"Transit Candidate (KIC {transit_sample['kepid']})")
plot_comparison(non_transit_sample, f"{non_transit_sample['label'].capitalize()} (KIC {non_transit_sample['kepid']})")

## 3. Extracting Transit Parameters

We can write a quick estimator function to calculate transit parameters like **Transit Depth**, **Stellar Noise**, and **Signal-to-Noise Ratio (SNR)** on the preprocessed signals.

In [ ]:
def estimate_transit_parameters(flux):
    """
    Calculates basic transit metrics on a normalized and detrended curve.
    """
    noise = estimate_noise(flux)
    
    # The transit depth is the maximum dip below the median baseline (0.0)
    depth = np.abs(np.min(flux))
    
    # Calculate Signal-to-Noise ratio (SNR)
    snr = depth / noise if noise > 0 else 0
    
    return {
        "transit_depth_percent": depth * 100,
        "estimated_noise": noise,
        "snr": snr
    }

for index, row in df[df["label"] == "transit"].head(5).iterrows():
    filepath = row["file"]
    if not os.path.exists(filepath):
        filepath = os.path.join("..", filepath)
        
    data = np.load(filepath)
    flux = align_sequence_length(preprocess_light_curve(data["flux"]), target_len=2000)
    
    params = estimate_transit_parameters(flux)
    print(f"KIC {row['kepid']} | Depth: {params['transit_depth_percent']:.4f}% | Noise: {params['estimated_noise']:.6f} | SNR: {params['snr']:.2f}")